# NeuroProfile on Colab


## 0 · GPU check

In [ ]:
import torch
try:
    from tribev2 import TribeModel
    import neuralset
    print(">>> IMPORT OK — tribev2 loads on torch", torch.__version__)
except Exception as e:
    import traceback; traceback.print_exc()
    print("\n>>> IMPORT FAILED:", type(e).__name__, "-", e)

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB, 81153 MiB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1 · Installs — run this, then **restart the runtime once**

All pip installs up front. After this cell: **Runtime ▸ Restart session**, then continue from Section 2. Restarting makes the pinned torch/numpy the versions that actually load (Colab preloads its own torch).

In [3]:
# numpy + torch must match tribev2's pins (numpy==2.2.6, torch>=2.5.1,<2.7).
# Installing the exact triple matches your handoff; if Colab's stock torch is already
# in [2.5.1, 2.7) you can skip the torch line and let tribev2 accept it.
!pip install -q numpy==2.2.6
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# TRIBE v2 (not on PyPI). Its pins match ours, so it won't swap the torch triple.
!pip install -q "git+https://github.com/facebookresearch/tribev2.git"

# CPU-side pipeline deps (your repo's requirements, listed for a clean Colab env)
!pip install -q nibabel qdrant-client fastapi python-multipart uvicorn yt-dlp pytest scipy

# whisperX installed IN THIS env so TRIBE calls it directly (patched below) instead of
# uvx re-downloading ~3.5 GB every call. If this bumps torch, re-run the torch line above.
!pip install -q whisperx

print("installs done — now Runtime > Restart session, then run Section 2")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 127.7 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.7 MB/s eta 0:00:0000:0100:01
ERROR: Could not find a version that satisfies the requirement torchvision==0.20.1 (from versions: 0.1.6, 0.2.0)
ERROR: No matching distribution found for torchvision==0.20.1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 67.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.1/258.1 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 12.5 M

### ⚠️ Restart the runtime now (Runtime ▸ Restart session), then run Section 2 onward.

## 2 · Drive, repo, auth, weights

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
NP = '/content/drive/MyDrive/neuroprofile'   # durable root (survives disconnects)
for sub in ('qdrant_data', 'data/timelines', 'clips'):
    os.makedirs(f'{NP}/{sub}', exist_ok=True)
print('durable root:', NP)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
durable root: /content/drive/MyDrive/neuroprofile


In [2]:
# Get the repo onto FAST local scratch (/content), not Drive (Drive FUSE is slow for code).
# Make sure ica/ (frozen artifacts) and tests/reducer_reference.npz come along —
# reducer.py does np.load("ica/...") at import time and will crash without them.

# --- Option A: clone from GitHub (fill in your remote) ---
!git pull
!git clone https://github.com/Mammbo/NeuroProfile.git /content/neuroprofile

# --- Option B: repo already in Drive — copy to scratch ---
# !cp -r /content/drive/MyDrive/neuroprofile/repo /content/neuroprofile

%cd /content/neuroprofile
!ls backend ica batch_encoding/

fatal: not a git repository (or any of the parent directories): .git
Cloning into '/content/neuroprofile'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 190 (delta 80), reused 172 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 811.17 KiB | 3.98 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/neuroprofile
backend:
app.py	    fetcher.py	 input_handler.py  stitcher.py
chunker.py  __init__.py  reducer.py	   storage.py

batch_encoding/:
analyze_server.py  chunk_runner.py    setup_gpu.sh
batch_encode.py    _encode_worker.py  test_encode.py

ica:
atlas			       README.md
fsaverage5_glasser_ids.json    region_system_map.json
fsaverage5_glasser_labels.npy


In [3]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Paste HF read token: ").strip()

# verify it works:
from huggingface_hub import HfApi
print("auth OK as", HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"])

auth OK as Mammbo


In [4]:
# Colab's network is fast — this is the step that swung 47min->5.5hr on the 4060.
!hf download meta-llama/Llama-3.2-3B --include "*.safetensors" "config.json" "tokenizer*"
# older huggingface_hub: !huggingface-cli download meta-llama/Llama-3.2-3B --include "*.safetensors" "config.json" "tokenizer*"

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]Downloading 'tokenizer_config.json' to '/root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B/blobs/cb9ec25536e44d86778b10509d3e5bdca459a5cf.incomplete'

config.json: 100% 844/844 [00:00<00:00, 5.25MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B/blobs/47d4a5aa69cdef91a53b77f5c5583647a578ca0e
Fetching 5 files:  20% 1/5 [00:00<00:02,  1.83it/s]
tokenizer_config.json: 50.5kB [00:00, 92.8MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B/blobs/cb9ec25536e44d86778b10509d3e5bdca459a5cf

model-00002-of-00002.safetensors:   0% 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0% 0.00/4.97G [00:00<?, ?B/s]


tokenizer.json: 0.00B [00:00, ?B/s]


tokenizer.json: 303kB [00:00, 3.03MB/s]


tokenizer.json: 634kB [00:00, 2.34MB/s]


tokenizer.json: 893kB [00:00, 2.42MB/s]


tokenizer.json: 1.29MB [00:00, 2.93MB/s]


tokenizer.json: 

In [5]:
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
import nltk
for r in ["punkt_tab", "punkt"]:
    nltk.download(r)

# Patch TRIBE so get_events_dataframe calls whisperx directly instead of via `uvx`
# (uvx spins an isolated env and re-downloads ~3.5 GB every call -> looks frozen at 0%).
import tribev2, pathlib
et  = pathlib.Path(tribev2.__file__).parent / "eventstransforms.py"
src = et.read_text()
new = src.replace('["uvx", "whisperx"', '["whisperx"')
et.write_text(new)
print("uvx->whisperx patch:", "applied" if new != src else "NO CHANGE — check the pattern in eventstransforms.py")

# NOTE: no ctranslate2 execstack ELF-patch needed on Colab (Ubuntu kernel allows exec-stack).
# TRIBE's default whisperx is large-v3 fp16 on cuda. On a 16 GB T4 it may fit alongside TRIBE.
# If predict()+whisperx OOM even here, apply your 4060 edit (model="small", device="cpu",
# compute_type="int8") to eventstransforms.py.

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.13/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-08-28 15:54:00 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


uvx->whisperx patch: NO CHANGE — check the pattern in eventstransforms.py


## 3 · Feed clips (upload `.mp4`s to Drive, then encode)

Upload your pre-downloaded clips to `MyDrive/neuroprofile/clips/`. The **file route** needs no yt-dlp, no cookies, no download — `resolve_source` sniffs magic bytes and goes straight to `chunk_video`.

In [6]:
!pip install -q ffmpeg-python

In [7]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuroprofile/clips', exist_ok=True)
!ls -la /content/drive/MyDrive/neuroprofile/clips

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 191461
-rw------- 1 root root  5468248 Aug 21 04:11 'Dog of Wisdom II [TnlakHr-O4w].mp4'
-rw------- 1 root root 28439109 Aug 21 05:48 'history of japan [Mh5LY4Mz15o].mp4'
-rw------- 1 root root 68840039 Aug 21 05:45 'history of the entire world, i guess [xuCn8ux2gbs].mp4'
-rw------- 1 root root  5458973 Aug 21 05:50 'How did the fall of Constantinople affect LeBron'\''s legacy？ [IBP5NUDP28A].mp4'
-rw------- 1 root root 16575426 Aug 21 05:46 'Prank interview with Elijah Wood [IfhMILe8C84].mp4'
-rw------- 1 root root 10546804 Aug 21 05:50 'Tame Impala - Loser (Official Video) [s3a4OQR-10M].mp4'
-rw------- 1 root root  2250717 Aug 21 04:06  test1.mp4
-rw------- 1 root root  3998826 Aug 21 05:54 'Victor Wembanyama blocks Lively then cooks him with nasty handles for 4-pt play 😭 [CkgubzZICHE].mp4'
-rw------- 1 root root 54475071 Aug 21 05:51 'Why no one guard

In [ ]:
%cd /content/neuroprofile
!python batch_encoding/test_encode.py /content/drive/MyDrive/neuroprofile/clips/test1.mp4

In [8]:
# Build a corpus of FILE PATHS (not URLs) and grind it, persisting to Drive.
import glob, pathlib
clips = sorted(glob.glob('/content/drive/MyDrive/neuroprofile/clips/*.mp4'))
pathlib.Path('/content/corpus.txt').write_text("\n".join(clips))
print(len(clips), "clips -> /content/corpus.txt")

9 clips -> /content/corpus.txt


In [ ]:
#TEST run with 2 clips 
%cd /content/neuroprofile
!python batch_encoding/batch_encode.py --corpus /content/corpus.txt --limit 2 \
    --qdrant-path   /content/drive/MyDrive/neuroprofile/qdrant_data \
    --timelines-dir /content/drive/MyDrive/neuroprofile/data/timelines


In [9]:
from qdrant_client import QdrantClient
QP = "/content/drive/MyDrive/neuroprofile/qdrant_data"
client = QdrantClient(path=QP)

print("collections:", [c.name for c in client.get_collections().collections])
print("count:", client.count("videos_v1").count)

pts, _ = client.scroll("videos_v1", limit=20, with_payload=True, with_vectors=False)
for p in pts:
    pl = p.payload
    print("\n—", pl["video_id"], "|", pl["title"], "| dur", pl.get("duration"), "s")
    print("   profile:", [round(x, 3) for x in pl["system_profile"]])
    print("   systems:", pl["system_names"])
    print("   moments:", len(pl.get("moments", [])), "| timeline:", pl["timeline_path"])

client.close()   # important — embedded Qdrant is single-process; close before anything else opens it

collections: ['videos_v1']
count: 7

— upload:e43505952add | Victor Wembanyama blocks Lively then cooks him with nasty handles for 4-pt play 😭 [CkgubzZICHE].mp4 | dur 48 s
   profile: [0.288, 0.151, 0.268, 0.099, 0.145, 0.032]
   systems: ['audiovisual_integration', 'social_sts_tpj', 'visual_motion', 'auditory', 'dmn_scene_medial_parietal', 'affect_reward']
   moments: 8 | timeline: upload_e43505952add.npz

— file:How did the fall of Constantinople affect LeBron's legacy？ [IBP5NUDP28A] | How did the fall of Constantinople affect LeBron's legacy？ [IBP5NUDP28A].mp4 | dur 220 s
   profile: [0.033, 0.016, 0.019, 0.06, 0.035, 0.026]
   systems: ['audiovisual_integration', 'social_sts_tpj', 'visual_motion', 'auditory', 'dmn_scene_medial_parietal', 'affect_reward']
   moments: 8 | timeline: 7066d85d9bf1.npz

— file:Dog of Wisdom II [TnlakHr-O4w] | Dog of Wisdom II [TnlakHr-O4w].mp4 | dur 220 s
   profile: [0.084, 0.054, 0.07, 0.043, 0.051, 0.029]
   systems: ['audiovisual_integration', 'socia

In [ ]:
!python batch_encoding/batch_encode.py --corpus /content/corpus.txt \
    --qdrant-path   /content/drive/MyDrive/neuroprofile/qdrant_data \
    --timelines-dir /content/drive/MyDrive/neuroprofile/data/timelines

# Persisted to Drive => a Colab disconnect mid-corpus is fine: re-run this cell and
# already-encoded ids are skipped (db.get_video != None -> status "skip"). 

live inference

In [ ]:
# 0.
%cd /content/neuroprofile
!git pull

# 1. kill the old server + tunnel if they're still running
!pkill -f analyze_server.py 2>/dev/null; pkill -f cloudflared 2>/dev/null
import time; time.sleep(2)

# 2. cloudflared (safe to re-run; no-op if already installed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# 3. start the backend 
import os, subprocess
subprocess.Popen(
    ["python","batch_encoding/analyze_server.py",
     "--qdrant-path","/content/drive/MyDrive/neuroprofile/qdrant_data",
     "--timelines-dir","/content/drive/MyDrive/neuroprofile/data/timelines",
     "--videos-dir","/content/drive/MyDrive/neuroprofile/videos",
     "--host","127.0.0.1","--port","8000"],
    cwd="/content/neuroprofile", env=dict(os.environ),
    stdout=open("/content/analyze.log","a"), stderr=subprocess.STDOUT)
time.sleep(8); print(open("/content/analyze.log").read()[-800:])

# 4. tunnel 
!cloudflared tunnel --url http://localhost:8000

/content/neuroprofile
Already up to date.
^C
                    media_type_for, resolve_video_file, slim, timeline_payload)
ModuleNotFoundError: No module named 'serving'
INFO:     Started server process [13009]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [13009]
[analyze] qdrant=/content/drive/MyDrive/neuroprofile/qdrant_data timelines=/content/drive/MyDrive/neuroprofile/data/timelines
INFO:     Started server process [13449]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)

2026-08-28T16:29:19Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. Howe